# 千葉県 自転車盗リスク可視化マップ（令和7年）
〜 市区町村ごとの認知件数と、無施錠・発生場所・時間帯から防犯を考える 〜

| 役割 | 氏名 | 担当 |
|---|---|---|
| メンバー1 | （小泉流駆） | （コード制作） |
| メンバー2 | （吉野健人　） | （アイデア、分析） |

- **問い**：千葉県内で自転車盗の認知件数が多い地域はどこか？ その地域では、どんな状況（無施錠・場所・時間帯）の被害が多いか？
- **使用データ**：千葉県警察公開「令和7年 自転車盗（CSV形式）」`bike.csv`（全10,054件）

## 分析の方針（注意点）
- 元データに緯度・経度はないため、**事件の地点は地図に出さず、市区町村単位の地域別集計**として表示する。
- 駐輪台数・自転車の保有台数（分母）がないため、「盗まれる確率」ではなく**認知件数の比較**として読む。
- 「無施錠率」は、被害にあった自転車のうち無施錠だった割合であり、「盗まれやすさ」そのものではない。
- 発生日が不明な76件・発生時が不明な103件は、**0時に変換せず**、時間帯の集計から除外する（除外件数は実行結果に表示）。
- 使い方：`bike.csv` をこのノートブックと同じフォルダに置き、上のセルから順に実行する。

In [1]:
# 必要なら最初に1回だけ実行（行頭の # を外す）
# %pip install folium pandas

import json
import os
import urllib.request
from html import escape

import folium
import pandas as pd

CSV_PATH = "bike.csv"
HTML_PATH = "chiba_bicycle_theft_map.html"
GEO_DIR = "geojson_chiba"   # 市区町村の境界データの保存先（2回目以降はここから読むので速い）

In [2]:
# ===== 1. データ読み込みと前処理 =====
df = pd.read_csv(CSV_PATH, encoding="cp932", encoding_errors="replace", dtype=str)
df.columns = df.columns.str.strip()

COL_CODE = "市区町村コード（発生地）"
COL_CITY = "市区町村（発生地）"
COL_KOBAN = "管轄交番・駐在所（発生地）"
COL_DATE = "発生年月日（始期）"
COL_HOUR = "発生時（始期）"
COL_PLACE = "発生場所"
COL_LOCK = "施錠関係"

need = [COL_CODE, COL_CITY, COL_KOBAN, COL_DATE, COL_HOUR, COL_PLACE, COL_LOCK]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"列が見つかりません: {missing}\n実際の列名: {list(df.columns)}")

df["code5"] = df[COL_CODE].str.strip().str[:5]          # JIS市区町村コード（先頭5桁）

# --- 不明データの扱い ---
# 発生日：数字が入っていないものを「不明」とする（今回の集計では日付を使わない）
n_date_unknown = int((~df[COL_DATE].str.contains(r"\d", na=False)).sum())
# 発生時：0〜23の数字以外は「不明」。0時に変換せず NaN のままにして集計から除く
hour = pd.to_numeric(df[COL_HOUR].str.extract(r"(\d+)")[0], errors="coerce")
df["hour"] = hour.where(hour.between(0, 23))
n_hour_unknown = int(df["hour"].isna().sum())

# 3時間ごとの時間帯（時刻不明は None）
df["band"] = df["hour"].map(
    lambda h: None if pd.isna(h) else f"{int(h) // 3 * 3}〜{int(h) // 3 * 3 + 2}時"
)

print(f"全件数: {len(df):,}件 / 市区町村数: {df['code5'].nunique()}")
print(f"発生日が不明: {n_date_unknown}件（課題文では76件。今回は日付を使わないので除外なし）")
print(f"発生時が不明: {n_hour_unknown}件（課題文では103件）→ 0時にはせず、時間帯の集計だけから除外")

全件数: 10,054件 / 市区町村数: 57
発生日が不明: 76件（課題文では76件。今回は日付を使わないので除外なし）
発生時が不明: 103件（課題文では103件）→ 0時にはせず、時間帯の集計だけから除外


In [3]:
# ===== 2. 市区町村ごとの集計（for文）とランキング =====
def top(series):
    """最も多い値と、その件数を返す（空なら「不明」）"""
    s = series.dropna().astype(str).str.strip()
    s = s[s != ""]
    if s.empty:
        return "不明", 0
    vc = s.value_counts()
    return vc.index[0], int(vc.iloc[0])


stats = {}                                   # {JISコード5桁: 集計結果}
for code, g in df.groupby("code5"):          # ← 市区町村ごとに繰り返す
    lock = g[COL_LOCK].str.strip()
    unlocked = int((lock == "施錠せず").sum())
    locked = int((lock == "施錠した").sum())
    place, place_n = top(g[COL_PLACE])
    band, band_n = top(g["band"])
    koban, koban_n = top(g[COL_KOBAN])
    stats[code] = {
        "name": top(g[COL_CITY])[0],
        "count": len(g),
        "unlocked": unlocked,
        "locked": locked,
        "rate": unlocked / len(g) * 100,     # 無施錠率（%）
        "place": place, "place_n": place_n,
        "band": band, "band_n": band_n,
        "koban": koban, "koban_n": koban_n,
        "hour_unknown": int(g["hour"].isna().sum()),
    }

TOTAL = sum(s["count"] for s in stats.values())   # 県全体の認知件数

# --- 4段階の色分け（下限, 凡例の文字, 色）。件数の分布に合わせて変えてもよい ---
LEVELS = [
    (500, "500件以上", "#b10026"),
    (200, "200〜499件", "#fc4e2a"),
    (100, "100〜199件", "#feb24c"),
    (1, "1〜99件", "#ffeda0"),
]
NO_DATA_COLOR = "#d9d9d9"


def level_of(n):
    """件数 → (ランクの文字, 色)"""
    for rank, (low, label, color) in enumerate(LEVELS, start=1):
        if n >= low:
            return f"ランク{rank}（{label}）", color
    return "データなし（0件）", NO_DATA_COLOR


# --- ランキング（上位10）と文字の棒グラフ ---
ranking = sorted(stats.items(), key=lambda kv: kv[1]["count"], reverse=True)
max_n = ranking[0][1]["count"]
print(f"【認知件数ランキング 上位10】（県全体 {TOTAL:,}件）")
for i, (code, s) in enumerate(ranking[:10], start=1):
    bar = "█" * max(1, round(s["count"] / max_n * 30))
    print(f"{i:>2}位 {s['name'].ljust(6, chr(12288))} {s['count']:>5,}件 {bar}")

# --- 各ランクに何市区町村あるか（分布の確認） ---
level_count = {}
for code, s in stats.items():
    key = level_of(s["count"])[0]
    level_count[key] = level_count.get(key, 0) + 1
print("\n【ランク別の市区町村数】")
for key, v in level_count.items():
    print(f"  {key}: {v}市区町村")

【認知件数ランキング 上位10】（県全体 10,054件）
 1位 船橋市　　　 1,299件 ██████████████████████████████
 2位 市川市　　　   871件 ████████████████████
 3位 松戸市　　　   725件 █████████████████
 4位 千葉市中央区   708件 ████████████████
 5位 柏市　　　　   564件 █████████████
 6位 習志野市　　   528件 ████████████
 7位 八千代市　　   455件 ███████████
 8位 浦安市　　　   407件 █████████
 9位 千葉市美浜区   370件 █████████
10位 千葉市稲毛区   358件 ████████

【ランク別の市区町村数】
  ランク1（500件以上）: 6市区町村
  ランク2（200〜499件）: 12市区町村
  ランク4（1〜99件）: 31市区町村
  ランク3（100〜199件）: 8市区町村


In [4]:
# ===== 3. 市区町村の境界データ（1市区町村=1ファイルを取得） =====
# ※ prefectures/12.json は「千葉県全体で1つの図形」なので、全域が同じ色になってしまう。
#    そこで geojson/12/<コード>.json を1市区町村ずつ取得して使う。
CHIBA_CODES = """
12101 12102 12103 12104 12105 12106 12202 12203 12204 12205 12206 12207 12208 12210 12211
12212 12213 12215 12216 12217 12218 12219 12220 12221 12222 12223 12224 12225 12226 12227
12228 12229 12230 12231 12232 12233 12234 12235 12236 12237 12238 12239 12322 12329 12342
12347 12349 12403 12409 12410 12421 12422 12423 12424 12426 12427 12441 12443 12463
""".split()
GEO_URL = "https://raw.githubusercontent.com/niiyz/JapanCityGeoJson/master/geojson/12/{}.json"
PRECISION = 3   # 座標の小数桁数。3=軽い（約100m刻み） / 4=細かいが約4倍重い


def thin(ring):
    """座標を丸めて、同じ点の連続を除く（HTMLを軽くする）"""
    pts = []
    for x, y in ring:
        p = [round(x, PRECISION), round(y, PRECISION)]
        if not pts or p != pts[-1]:
            pts.append(p)
    return pts


def simplify(geometry):
    polys = geometry["coordinates"] if geometry["type"] == "MultiPolygon" else [geometry["coordinates"]]
    out = []
    for poly in polys:
        rings = [thin(r) for r in poly]
        if len(rings[0]) >= 4:                       # 外側の輪が潰れた極小の島は捨てる
            out.append([r for r in rings if len(r) >= 4])
    return {"type": "MultiPolygon", "coordinates": out}


os.makedirs(GEO_DIR, exist_ok=True)
features = []
for code in CHIBA_CODES:                             # ← 市区町村ごとに取得
    path = os.path.join(GEO_DIR, f"{code}.json")
    if not os.path.exists(path):
        req = urllib.request.Request(GEO_URL.format(code), headers={"User-Agent": "Mozilla/5.0"})
        data = urllib.request.urlopen(req, timeout=30).read()
        with open(path, "wb") as f:
            f.write(data)
    with open(path, encoding="utf-8") as f:
        ft = json.load(f)["features"][0]
    p = ft["properties"]
    features.append({
        "code": code,
        "geo_name": p["N03_003"] + p["N03_004"],     # 例：千葉市中央区 / 柏市
        "geometry": simplify(ft["geometry"]),
    })

print(f"境界データ: {len(features)}市区町村分を読み込みました")
no_geo = [c for c in stats if c not in CHIBA_CODES]
if no_geo:
    print("⚠ 境界データがないコード（地図に出ません）:", no_geo)

境界データ: 59市区町村分を読み込みました


In [5]:
# ===== 4. 地図の作成（色分け・ポップアップ・ホバー・凡例） =====
def advice_of(rate):
    """無施錠率に応じた防犯アドバイス"""
    if rate >= 70:
        return "無施錠での被害が7割以上です。短時間の駐輪でも必ず鍵をかけ、前輪・後輪の2か所を施錠（ダブルロック）しましょう。"
    if rate >= 50:
        return "被害の半数以上が無施錠です。「ちょっとだけ」でも施錠する習慣を。駐輪場では支柱などに固定（地球ロック）すると効果的です。"
    if rate >= 30:
        return "無施錠の被害が目立ちます。施錠していても被害はあるため、頑丈なU字ロックなど2つ目の鍵を足しましょう。"
    return "施錠していても被害が出ています。細いワイヤー錠だけに頼らず、ダブルロックと、人目・照明のある駐輪場の利用を。"


def make_popup(name, s, rank_text, color):
    e = escape
    return f"""
    <div style="font-family:'Meiryo','Yu Gothic',sans-serif;font-size:13px;line-height:1.7;min-width:270px">
      <div style="font-size:16px;font-weight:bold">{e(name)}</div>
      <div><span style="display:inline-block;width:14px;height:14px;background:{color};border:1px solid #555;vertical-align:middle"></span>
        件数ランク：<b>{e(rank_text)}</b></div>
      <hr style="margin:6px 0">
      <b>認知件数</b>：{s['count']:,}件（県全体の{s['count'] / TOTAL * 100:.1f}%）<br>
      <b>無施錠</b>：{s['unlocked']:,}件 ／ 施錠あり：{s['locked']:,}件<br>
      <b>無施錠率</b>：{s['rate']:.1f}%<br>
      <b>最多の発生場所</b>：{e(s['place'])}（{s['place_n']}件）<br>
      <b>最多の時間帯</b>：{e(s['band'])}（{s['band_n']}件）<br>
      <b>最多の交番・駐在所</b>：{e(s['koban'])}（{s['koban_n']}件）<br>
      <hr style="margin:6px 0">
      <b>防犯アドバイス</b><br>{advice_of(s['rate'])}
      <div style="color:#777;font-size:11px;margin-top:4px">※時間帯は、発生時が不明の{s['hour_unknown']}件を除いて集計</div>
    </div>"""


m = folium.Map(location=[35.5, 140.2], zoom_start=9, tiles=None)
folium.TileLayer(                                     # 国土地理院タイル（APIキー不要）
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank">国土地理院</a>',
    name="国土地理院 淡色地図",
    max_zoom=18,
).add_to(m)

for ft in features:                                   # ← 1市区町村=1レイヤーで、色を1つずつ直接指定
    s = stats.get(ft["code"])
    if s:
        name = s["name"]
        rank_text, color = level_of(s["count"])
        tip = f"{name}：{s['count']:,}件"
        popup_html = make_popup(name, s, rank_text, color)
    else:                                             # データに無い市区町村（0件）
        name = ft["geo_name"]
        rank_text, color = level_of(0)
        tip = f"{name}：0件（データなし）"
        popup_html = f"<b>{escape(name)}</b><br>{rank_text}"

    layer = folium.GeoJson(
        {"type": "Feature", "properties": {}, "geometry": ft["geometry"]},
        # color=color で「今の色」をその場で固定する（これが全域同色になる不具合の回避策）
        style_function=lambda _f, color=color: {
            "fillColor": color, "color": "#555555", "weight": 1, "fillOpacity": 0.8,
        },
        highlight_function=lambda _f: {"weight": 3, "color": "#000000"},
    )
    folium.Tooltip(tip, sticky=True).add_to(layer)          # ホバー
    folium.Popup(popup_html, max_width=340).add_to(layer)   # クリック／タップ
    layer.add_to(m)

# --- 凡例（左下） ---
def swatch(c):
    return f'<span style="display:inline-block;width:18px;height:12px;background:{c};border:1px solid #555;margin-right:6px;vertical-align:middle"></span>'

legend_rows = "".join(f"<div>{swatch(c)}{label}</div>" for _, label, c in LEVELS)
legend_rows += f"<div>{swatch(NO_DATA_COLOR)}0件（データなし）</div>"
legend_html = f"""
<div style="position:fixed;bottom:24px;left:10px;z-index:9999;background:rgba(255,255,255,.92);
            padding:8px 12px;border:1px solid #888;border-radius:6px;font-size:12px;line-height:1.7;
            font-family:'Meiryo','Yu Gothic',sans-serif">
  <b>自転車盗 認知件数</b><br>{legend_rows}
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

m.fit_bounds([[34.89, 139.73], [36.11, 140.89]])           # 千葉県全体が入るように
m.save(HTML_PATH)
print(f"保存しました: {os.path.abspath(HTML_PATH)}（ブラウザで開けます）")
m

保存しました: [local]\OneDrive\ドキュメント\Programming\week8\baik.challeng\chiba_bicycle_theft_map.html（ブラウザで開けます）


In [6]:
# ===== 5. 考察に使う数値（この出力を見て、下のMarkdownセルを書く） =====
MIN_N = 30   # 無施錠率の比較は、件数が少ない市区町村を除く（偶然のぶれが大きいため）

all_unlocked = sum(s["unlocked"] for s in stats.values())
print(f"県全体: {TOTAL:,}件 / 無施錠 {all_unlocked:,}件（無施錠率 {all_unlocked / TOTAL * 100:.1f}%）")

top3 = ranking[:3]
top3_sum = sum(s["count"] for _, s in top3)
print("件数上位3: " + "、".join(f"{s['name']} {s['count']:,}件" for _, s in top3)
      + f"（3市区町村で県全体の{top3_sum / TOTAL * 100:.1f}%）")

big = [(c, s) for c, s in stats.items() if s["count"] >= MIN_N]      # 絞り込み
hi = max(big, key=lambda kv: kv[1]["rate"])[1]
lo = min(big, key=lambda kv: kv[1]["rate"])[1]
print(f"無施錠率（{MIN_N}件以上の{len(big)}市区町村の中で）最高: {hi['name']} {hi['rate']:.1f}%（{hi['count']}件）"
      f" / 最低: {lo['name']} {lo['rate']:.1f}%（{lo['count']}件）")

place, place_n = top(df[COL_PLACE])
print(f"県全体で最多の発生場所: {place}（{place_n:,}件＝全体の{place_n / len(df) * 100:.1f}%）")

# --- 時間帯別の件数（文字の棒グラフ。発生時が不明の件は含めない） ---
known = len(df) - n_hour_unknown
print(f"\n【時間帯別の認知件数】（発生時が分かる{known:,}件。不明{n_hour_unknown}件は除外）")
band_labels = [f"{h}〜{h + 2}時" for h in range(0, 24, 3)]
band_counts = {}
for label in band_labels:                                             # ← 時間帯ごとに数える
    band_counts[label] = int((df["band"] == label).sum())
peak = max(band_counts.values())
for label, n in band_counts.items():
    print(f"{label.rjust(7)} {n:>5,}件 ({n / known * 100:4.1f}%) {'█' * round(n / peak * 30)}")

県全体: 10,054件 / 無施錠 6,879件（無施錠率 68.4%）
件数上位3: 船橋市 1,299件、市川市 871件、松戸市 725件（3市区町村で県全体の28.8%）
無施錠率（30件以上の34市区町村の中で）最高: 旭市 86.4%（66件） / 最低: 八街市 55.0%（129件）
県全体で最多の発生場所: その他（3,297件＝全体の32.8%）

【時間帯別の認知件数】（発生時が分かる9,951件。不明103件は除外）
   0〜2時   289件 ( 2.9%) ████
   3〜5時   197件 ( 2.0%) ███
   6〜8時 1,966件 (19.8%) █████████████████████████████
  9〜11時 1,256件 (12.6%) ███████████████████
 12〜14時 1,337件 (13.4%) ████████████████████
 15〜17時 2,036件 (20.5%) ██████████████████████████████
 18〜20時 1,947件 (19.6%) █████████████████████████████
 21〜23時   923件 ( 9.3%) ██████████████


### 考察

### 数値から分かったこと
1. **認知件数は一部の市区町村に集中している。** 船橋市が1,299件で最多（県全体の12.9%）。上位3市（船橋市・市川市・松戸市）で2,895件、県全体の28.8%を占める。500件以上のランク1は6市区町村（船橋市・市川市・松戸市・千葉市中央区・柏市・習志野市）で、合計4,695件（46.7%）。57市区町村のうち約1割で、県全体の約半分になる。
2. **被害の約7割が無施錠だった。** 県全体の無施錠率は68.4%（6,879件／10,054件）。件数が30件以上の34市区町村でも、最も低い八街市で55.0%、最も高い旭市で86.4%。地域によらず、無施錠が過半数を占めた。

### 防犯の提案
時間帯別では、6〜8時・15〜17時・18〜20時の3つで5,949件（発生時が分かる9,951件の59.8%）を占め、0〜5時は486件（4.9%）だった。通勤・通学や帰宅の時間帯に自転車を止める人が多いためと考えられる。また、28市区町村で「駐車（輪）場」が発生場所の最多だった。そこで、**駐車（輪）場に止めるときは、短時間でも必ず施錠し、前輪と後輪の2か所を施錠する（ダブルロック）**ことを提案する。特に件数の多い船橋市・市川市・松戸市で優先して呼びかけたい。

### 注意（この分析の限界）
- 市区町村ごとの**認知件数の比較**であり、人口・自転車の台数・駐輪台数で割っていない。件数が多い地域が「盗まれる確率が高い」とは言えない。
- 無施錠率は「被害にあった自転車のうち無施錠だった割合」であり、止めてある自転車全体の施錠状況は分からない。
- 発生時は「始期」のため、盗まれた時刻そのものとは限らない。発生時が不明な103件は時間帯の集計から除外した（0時には含めていない）。
- 元データに緯度・経度がないため、事件の地点ではなく市区町村単位で示している。